# 04 — Pipeline Orchestrator

Drives the full **Bronze → Silver → Gold** medallion pipeline by calling notebooks 02, 03, and 05
via `dbutils.notebook.run`.  Four clearly-labelled sections:

| Section | Purpose |
|---------|---------|
| **A — STANDARD INCREMENTAL RUN** | Normal nightly cadence for all repos / all entities / Gold star schema |
| **B — BACKFILL RUN** | Re-process an explicit time window for one repo and refresh Gold |
| **C — DRIFT TEST** | Exercises schema-drift detection, Silver quarantine, and Gold resilience |
| **D — IDEMPOTENCY PROOF** | Runs the same batch twice; asserts Bronze, Silver, and Gold counts & measures are identical |

Each section ends with a **verification cell** that prints row counts across all layers and the
last 15 rows of `ops.pipeline_execution_logs`.

## ── Global Widgets & Helpers ──

In [0]:
dbutils.widgets.text("catalog",        "workspace",                         "Unity Catalog Name")
dbutils.widgets.text("base_path",      "/Volumes/workspace/bronze_data/raw", "Volume Base Path")
dbutils.widgets.text("batch_id",       "",                                   "Batch ID (empty = auto)")
dbutils.widgets.text("nb_02_path",     "./02_raw_to_bronze",                 "Path to notebook 02")
dbutils.widgets.text("nb_03_path",     "./03_bronze_to_silver",              "Path to notebook 03")
dbutils.widgets.text("nb_05_path",     "./05_silver_to_gold",                "Path to notebook 05")
dbutils.widgets.dropdown("gold_mode",  "incremental", ["incremental", "full"], "Gold Processing Mode")
dbutils.widgets.text("timeout_s",      "3600",                               "Notebook timeout (seconds)")
# Section A/D date window — override these to change which window is fetched
dbutils.widgets.text("incr_since",     "2026-09-25T00:00:00Z",               "Section A+D: incremental since")
dbutils.widgets.text("incr_until",     "",                                   "Section A+D: incremental until (empty = unbounded)")
# Section B backfill parameters
dbutils.widgets.text("backfill_repo",  "surrealdb/surrealdb",                "Section B: repo for backfill")
dbutils.widgets.text("backfill_since", "2026-04-01T00:00:00Z",               "Section B: backfill window start")
dbutils.widgets.text("backfill_until", "2026-06-30T23:59:59Z",               "Section B: backfill window end")
dbutils.widgets.text("salt", "", "PII salt (used if no secret scope)")
dbutils.widgets.dropdown("run_ingestion", "false", ["false", "true"], "Run GitHub API Ingestion before Section A")

In [0]:
import uuid
from datetime import datetime, timezone

CATALOG        = dbutils.widgets.get("catalog")
BASE_PATH      = dbutils.widgets.get("base_path")
NB_02          = dbutils.widgets.get("nb_02_path")
NB_03          = dbutils.widgets.get("nb_03_path")
NB_05          = dbutils.widgets.get("nb_05_path")
GOLD_MODE      = dbutils.widgets.get("gold_mode")
TIMEOUT        = int(dbutils.widgets.get("timeout_s"))
BATCH_ID       = dbutils.widgets.get("batch_id") or f"run_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}_{uuid.uuid4().hex[:6]}"
# Section-scoped date params (read from widgets — override in the widget bar)
INCR_SINCE     = dbutils.widgets.get("incr_since")
INCR_UNTIL     = dbutils.widgets.get("incr_until")
BACKFILL_REPO  = dbutils.widgets.get("backfill_repo")
BACKFILL_SINCE = dbutils.widgets.get("backfill_since")
BACKFILL_UNTIL = dbutils.widgets.get("backfill_until")

ALL_REPOS = [
    "postgres/postgres",
    "cockroachdb/cockroach",
    "mongodb/mongo",
    "surrealdb/surrealdb",
    "redis/redis",
    "apache/cassandra",
    "facebookresearch/faiss",
    "qdrant/qdrant",
    "milvus-io/milvus",
    "chroma-core/chroma",
]
ALL_ENTITIES = ["repo_metadata", "commits", "issues", "pulls", "releases", "stars_history"]

print(f"Orchestrator ready")
print(f"  catalog   : {CATALOG}")
print(f"  base_path : {BASE_PATH}")
print(f"  batch_id  : {BATCH_ID}")
print(f"  NB_02     : {NB_02}")
print(f"  NB_03     : {NB_03}")
print(f"  NB_05     : {NB_05}")
print(f"  gold_mode : {GOLD_MODE}")
print(f"  timeout   : {TIMEOUT}s")

Orchestrator ready
  catalog   : workspace
  base_path : /Volumes/workspace/bronze_data/raw
  batch_id  : run_20261117T184549Z_4692b9
  NB_02     : ./02_raw_to_bronze
  NB_03     : ./03_bronze_to_silver
  NB_05     : ./05_silver_to_gold
  gold_mode : incremental
  timeout   : 3600s


In [0]:
# Helper: run a single notebook, return its exit value, and print timing
def run_nb(path, params, timeout=TIMEOUT):
    """Thin wrapper around dbutils.notebook.run with timing and error surfacing."""
    if path == NB_03:
        params = {**params, "salt": dbutils.widgets.get("salt")}
    started = datetime.now(timezone.utc)
    result = dbutils.notebook.run(path, timeout, params)
    elapsed = (datetime.now(timezone.utc) - started).total_seconds()
    print(f"  ✓ {path.split('/')[-1]}  → {result!r}  ({elapsed:.0f}s)")
    return result

In [0]:
# Helper: show row counts for all Gold + Silver + Bronze tables
def show_counts(label=""):
    print(f"\n{'='*75}")
    print(f"ROW COUNTS  {label}")
    print(f"{'='*75}")
    layers = [
        (f"{CATALOG}.bronze", ["commits", "issues", "pull_requests", "releases", "repo_metadata", "stars_history"]),
        (f"{CATALOG}.silver", ["commits", "issues", "pull_requests", "releases", "repo_metadata", "stars_weekly"]),
        (f"{CATALOG}.gold",   [
            "dim_repo", "dim_date", "dim_actor",
            "fact_commits", "fact_issues", "fact_pull_requests",
            "fact_releases", "fact_repo_snapshot", "fact_repo_daily_activity", "fact_repo_stars_weekly",
            "audit_gold_reconciliation"
        ]),
        (f"{CATALOG}.ops",    ["silver_quarantine", "pipeline_execution_logs"]),
    ]
    for schema, tables in layers:
        for tbl in tables:
            try:
                n = spark.table(f"{schema}.{tbl}").count()
                print(f"  {schema}.{tbl:<30} {n:>8,} rows")
            except Exception as e:
                print(f"  {schema}.{tbl:<30}  [not found: {e}]")

In [0]:
# Helper: print last 15 rows of ops.pipeline_execution_logs
def show_logs(label="", n=15):
    print(f"\n{'─'*75}")
    print(f"Last {n} rows of ops.pipeline_execution_logs  {label}")
    print(f"{'─'*75}")
    try:
        cols = [
            "log_id", "layer", "status", "start_time",
            "rows_inserted", "rows_updated", "rows_aggregated", "parameter"
        ]
        available = [c for c in cols if c in spark.table(f"{CATALOG}.ops.pipeline_execution_logs").columns]
        (spark.table(f"{CATALOG}.ops.pipeline_execution_logs")
              .select(*available)
              .orderBy("start_time", ascending=False)
              .limit(n)
              .show(truncate=80))
    except Exception as e:
        print(f"  [logs table not found or empty: {e}]")

### Pre-flight: Validate Salt Configuration
Ensures that PII salt is resolvable (via secrets, widget, or environment variable) before running pipeline notebooks.

In [0]:
import sys
import os

workspace_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if workspace_root not in sys.path:
    sys.path.append(workspace_root)

from src.common import get_salt
from src.pipeline_preflight import ensure_source_layout

try:
    _configured_salt = get_salt(dbutils)
    print(f"  ✓ Salt verified and active: length = {len(_configured_salt)} characters")
except Exception as _salt_err:
    raise ValueError(f"Pre-flight failed: {_salt_err}")


def _source_file_exists(path):
    """Check that a Volume file exists; empty JSON arrays are valid no-data inputs."""
    try:
        stats = dbutils.fs.ls(path)
        return bool(stats)
    except Exception:
        return False


def _run_incremental_ingestion():
    """Land a complete incremental source batch and return its root path."""
    landing_root = "/Volumes/workspace/bronze_data/incremental"
    ingest_batch = f"{BATCH_ID}_incr"
    print(f"[Ingestion Hook] Running ./00_ingest_github_api in incremental mode for batch {ingest_batch}...")
    run_nb("./00_ingest_github_api", {
        "catalog":      CATALOG,
        "repo":         "ALL",
        "entity":       "ALL",
        "run_mode":     "incremental",
        "since":        INCR_SINCE,
        "until":        INCR_UNTIL,
        "batch_id":     ingest_batch,
        "landing_root": landing_root,
    })
    return f"{landing_root}/{ingest_batch}"


  [Warning] PII salt is shorter than 16 characters; consider using a longer salt for production security.
  ✓ Salt verified and active: length = 7 characters


In [0]:
# ── Optional Pre-Ingestion Hook ─────────────────────────────────────────────
# When widget run_ingestion is "true", runs ./00_ingest_github_api in incremental mode
# and points Section A's Bronze run to the landed batch path.
if dbutils.widgets.get("run_ingestion").strip().lower() in ("true", "1", "yes"):
    BASE_PATH = _run_incremental_ingestion()
    print(f"[Ingestion Hook] Pointing Section A Bronze run to base_path: {BASE_PATH}")

---
## ══════════════════════════════════════════════════════
## SECTION A — STANDARD INCREMENTAL RUN
## ══════════════════════════════════════════════════════
**Purpose:** Nightly incremental ingest — picks up everything updated since
`since` for all 10 repos and all 5 entities, then promotes to Silver.

In [0]:
# ── A-1 : Configure incremental window ──────────────────────────────────────
# Dates driven by widgets 'incr_since' and 'incr_until' set in the widget bar above.
A_SINCE     = INCR_SINCE          # widget: incr_since  (e.g. 2026-09-25T00:00:00Z)
A_UNTIL     = INCR_UNTIL          # widget: incr_until  (empty = no upper bound)
A_RUN_MODE  = "incremental"
A_BATCH_ID  = f"{BATCH_ID}_incr"

print(f"[Section A] STANDARD INCREMENTAL RUN")
print(f"  since     : {A_SINCE}")
print(f"  until     : {A_UNTIL or '(none)'}")
print(f"  run_mode  : {A_RUN_MODE}")
print(f"  batch_id  : {A_BATCH_ID}")

[Section A] STANDARD INCREMENTAL RUN
  since     : 2026-09-25T00:00:00Z
  until     : (none)
  run_mode  : incremental
  batch_id  : run_20261117T184549Z_4692b9_incr


In [0]:
# ── A-2 : Bronze pass — all repos × all entities ─────────────────────────────
BASE_PATH = ensure_source_layout(
    BASE_PATH,
    ALL_REPOS,
    ALL_ENTITIES,
    _source_file_exists,
    _run_incremental_ingestion,
)
print(f"[A-2 pre-flight] Source layout validated: {BASE_PATH}")
print(f"\n[A-2] Bronze — all repos")
for repo in ALL_REPOS:
    for entity in ALL_ENTITIES:
        run_nb(NB_02, {
            "catalog":   CATALOG,
            "repo":      repo,
            "entity":    entity,
            "base_path": BASE_PATH,
            "since":     A_SINCE,
            "until":     A_UNTIL,
            "batch_id":  A_BATCH_ID,
            "run_mode":  A_RUN_MODE,
        })

[A-2 pre-flight] Source layout validated: /Volumes/workspace/bronze_data/raw

[A-2] Bronze — all repos
  ✓ 02_raw_to_bronze  → '{"batch_id": "run_20261117T184549Z_4692b9_incr", "total_inserted": 0, "total_updated": 1, "results": [{"repo": "postgres/postgres", "entity": "repo_metadata", "status": "SUCCESS", "inserted": 0, "updated": 1}]}'  (72s)
  ✓ 02_raw_to_bronze  → '{"batch_id": "run_20261117T184549Z_4692b9_incr", "total_inserted": 417, "total_updated": 144, "results": [{"repo": "postgres/postgres", "entity": "commits", "status": "SUCCESS", "inserted": 417, "updated": 144}]}'  (31s)
  ✓ 02_raw_to_bronze  → '{"batch_id": "run_20261117T184549Z_4692b9_incr", "total_inserted": 0, "total_updated": 0, "results": [{"repo": "postgres/postgres", "entity": "issues", "status": "SUCCESS", "inserted": 0, "updated": 0}]}'  (21s)
  ✓ 02_raw_to_bronze  → '{"batch_id": "run_20261117T184549Z_4692b9_incr", "total_inserted": 0, "total_updated": 0, "results": [{"repo": "postgres/postgres", "entity": "pu

In [0]:
# ── A-3 : Silver pass — promote all repos ────────────────────────────────────
print(f"\n[A-3] Silver — all repos")
for repo in ALL_REPOS:
    run_nb(NB_03, {
        "catalog":  CATALOG,
        "repo":     repo,
        "since":    A_SINCE,
        "until":    A_UNTIL,
        "batch_id": A_BATCH_ID,
        "run_mode": A_RUN_MODE,
    })


[A-3] Silver — all repos
  ✓ 03_bronze_to_silver  → '{"batch_id": "run_20261117T184549Z_4692b9_incr", "records": [{"repository": "postgres/postgres", "entity": "commits", "bronze_rows": 561, "silver_inserted": 417, "silver_updated": 144, "quarantined_rows": 0, "excluded_prs": 0, "reconciled": true}, {"repository": "postgres/postgres", "entity": "issues", "bronze_rows": 0, "silver_inserted": 0, "silver_updated": 0, "quarantined_rows": 0, "excluded_prs": 0, "reconciled": true}, {"repository": "postgres/postgres", "entity": "pulls", "bronze_rows": 0, "silver_inserted": 0, "silver_updated": 0, "quarantined_rows": 0, "excluded_prs": 0, "reconciled": true}, {"repository": "postgres/postgres", "entity": "releases", "bronze_rows": 0, "silver_inserted": 0, "silver_updated": 0, "quarantined_rows": 0, "excluded_prs": 0, "reconciled": true}, {"repository": "postgres/postgres", "entity": "repo_metadata", "bronze_rows": 1, "silver_inserted": 0, "silver_updated": 1, "quarantined_rows": 0, "excluded_

In [0]:
# ── A-4 : Gold pass — star schema & aggregates ──────────────────────────────
print(f"\n[A-4] Gold — star schema & aggregates")
run_nb(NB_05, {
    "catalog":        CATALOG,
    "batch_id":       A_BATCH_ID,
    "gold_mode":      "incremental",
    "analysis_since": "2026-04-01",
    "analysis_until": "2026-09-30",
    "run_optimize":   "true",
})


[A-4] Gold — star schema & aggregates
  ✓ 05_silver_to_gold  → '{"batch_id": "run_20261117T184549Z_4692b9_incr", "status": "SUCCESS", "gold_mode": "incremental", "tables_processed": ["dim_repo", "dim_date", "dim_actor", "fact_commits", "fact_issues", "fact_pull_requests", "fact_releases", "fact_repo_snapshot", "fact_repo_daily_activity", "fact_repo_stars_weekly"], "quality_gates": "ALL_PASSED", "reconciliation": [{"table": "fact_commits", "silver_rows": 19042, "gold_rows": 19042, "delta": 0, "status": "PASS"}, {"table": "fact_issues", "silver_rows": 21751, "gold_rows": 21751, "delta": 0, "status": "PASS"}, {"table": "fact_pull_requests", "silver_rows": 13426, "gold_rows": 13426, "delta": 0, "status": "PASS"}, {"table": "fact_releases", "silver_rows": 795, "gold_rows": 795, "delta": 0, "status": "PASS"}, {"table": "fact_repo_snapshot", "silver_rows": 10, "gold_rows": 10, "delta": 0, "status": "PASS"}]}'  (314s)


'{"batch_id": "run_20261117T184549Z_4692b9_incr", "status": "SUCCESS", "gold_mode": "incremental", "tables_processed": ["dim_repo", "dim_date", "dim_actor", "fact_commits", "fact_issues", "fact_pull_requests", "fact_releases", "fact_repo_snapshot", "fact_repo_daily_activity", "fact_repo_stars_weekly"], "quality_gates": "ALL_PASSED", "reconciliation": [{"table": "fact_commits", "silver_rows": 19042, "gold_rows": 19042, "delta": 0, "status": "PASS"}, {"table": "fact_issues", "silver_rows": 21751, "gold_rows": 21751, "delta": 0, "status": "PASS"}, {"table": "fact_pull_requests", "silver_rows": 13426, "gold_rows": 13426, "delta": 0, "status": "PASS"}, {"table": "fact_releases", "silver_rows": 795, "gold_rows": 795, "delta": 0, "status": "PASS"}, {"table": "fact_repo_snapshot", "silver_rows": 10, "gold_rows": 10, "delta": 0, "status": "PASS"}]}'

In [0]:
# ── A-5 : Verification ───────────────────────────────────────────────────────
show_counts("after SECTION A — STANDARD INCREMENTAL RUN")
show_logs("after SECTION A", n=15)


ROW COUNTS  after SECTION A — STANDARD INCREMENTAL RUN
  workspace.bronze.commits                          19,042 rows
  workspace.bronze.issues                           40,229 rows
  workspace.bronze.pull_requests                    13,426 rows
  workspace.bronze.releases                            795 rows
  workspace.bronze.repo_metadata                        10 rows
  workspace.bronze.stars_history                     5,981 rows
  workspace.silver.commits                          19,042 rows
  workspace.silver.issues                           21,751 rows
  workspace.silver.pull_requests                    13,426 rows
  workspace.silver.releases                            795 rows
  workspace.silver.repo_metadata                        10 rows
  workspace.silver.stars_weekly                      5,981 rows
  workspace.gold.dim_repo                             10 rows
  workspace.gold.dim_date                          7,305 rows
  workspace.gold.dim_actor                         3

---
## ══════════════════════════════════════════════════════
## SECTION B — BACKFILL RUN
## ══════════════════════════════════════════════════════
**Purpose:** Re-process a specific historical window for a single repo.
Useful after fixing a bug in the transformation logic or after receiving
delayed data from the API.

In [0]:
# ── B-1 : Configure backfill window ─────────────────────────────────────────
# All three values driven by widgets: backfill_repo, backfill_since, backfill_until.
B_REPO      = BACKFILL_REPO        # widget: backfill_repo
B_SINCE     = BACKFILL_SINCE       # widget: backfill_since
B_UNTIL     = BACKFILL_UNTIL       # widget: backfill_until
B_RUN_MODE  = "backfill"
B_BATCH_ID  = f"{BATCH_ID}_backfill"

print(f"[Section B] BACKFILL RUN")
print(f"  repo      : {B_REPO}")
print(f"  since     : {B_SINCE}")
print(f"  until     : {B_UNTIL}")
print(f"  run_mode  : {B_RUN_MODE}")
print(f"  batch_id  : {B_BATCH_ID}")

[Section B] BACKFILL RUN
  repo      : surrealdb/surrealdb
  since     : 2026-04-01T00:00:00Z
  until     : 2026-06-30T23:59:59Z
  run_mode  : backfill
  batch_id  : run_20261117T184549Z_4692b9_backfill


In [0]:
# ── B-2 : Bronze pass — one repo, all entities ───────────────────────────────
print(f"\n[B-2] Bronze backfill — {B_REPO}")
for entity in ALL_ENTITIES:
    run_nb(NB_02, {
        "catalog":   CATALOG,
        "repo":      B_REPO,
        "entity":    entity,
        "base_path": BASE_PATH,
        "since":     B_SINCE,
        "until":     B_UNTIL,
        "batch_id":  B_BATCH_ID,
        "run_mode":  B_RUN_MODE,
    })


[B-2] Bronze backfill — surrealdb/surrealdb
  ✓ 02_raw_to_bronze  → '{"batch_id": "run_20261117T184549Z_4692b9_backfill", "total_inserted": 0, "total_updated": 0, "results": [{"repo": "surrealdb/surrealdb", "entity": "repo_metadata", "status": "SUCCESS", "inserted": 0, "updated": 0}]}'  (31s)
  ✓ 02_raw_to_bronze  → '{"batch_id": "run_20261117T184549Z_4692b9_backfill", "total_inserted": 0, "total_updated": 393, "results": [{"repo": "surrealdb/surrealdb", "entity": "commits", "status": "SUCCESS", "inserted": 0, "updated": 393}]}'  (31s)
  ✓ 02_raw_to_bronze  → '{"batch_id": "run_20261117T184549Z_4692b9_backfill", "total_inserted": 0, "total_updated": 177, "results": [{"repo": "surrealdb/surrealdb", "entity": "issues", "status": "SUCCESS", "inserted": 0, "updated": 177}]}'  (31s)
  ✓ 02_raw_to_bronze  → '{"batch_id": "run_20261117T184549Z_4692b9_backfill", "total_inserted": 0, "total_updated": 44, "results": [{"repo": "surrealdb/surrealdb", "entity": "pulls", "status": "SUCCESS", "inser

In [0]:
# ── B-3 : Silver pass — one repo ─────────────────────────────────────────────
print(f"\n[B-3] Silver backfill — {B_REPO}")
run_nb(NB_03, {
    "catalog":  CATALOG,
    "repo":     B_REPO,
    "since":    B_SINCE,
    "until":    B_UNTIL,
    "batch_id": B_BATCH_ID,
    "run_mode": B_RUN_MODE,
})


[B-3] Silver backfill — surrealdb/surrealdb
  ✓ 03_bronze_to_silver  → '{"batch_id": "run_20261117T184549Z_4692b9_backfill", "records": [{"repository": "surrealdb/surrealdb", "entity": "commits", "bronze_rows": 393, "silver_inserted": 0, "silver_updated": 393, "quarantined_rows": 0, "excluded_prs": 0, "reconciled": true}, {"repository": "surrealdb/surrealdb", "entity": "issues", "bronze_rows": 177, "silver_inserted": 0, "silver_updated": 100, "quarantined_rows": 0, "excluded_prs": 77, "reconciled": true}, {"repository": "surrealdb/surrealdb", "entity": "pulls", "bronze_rows": 44, "silver_inserted": 0, "silver_updated": 44, "quarantined_rows": 0, "excluded_prs": 0, "reconciled": true}, {"repository": "surrealdb/surrealdb", "entity": "releases", "bronze_rows": 9, "silver_inserted": 0, "silver_updated": 9, "quarantined_rows": 0, "excluded_prs": 0, "reconciled": true}, {"repository": "surrealdb/surrealdb", "entity": "repo_metadata", "bronze_rows": 0, "silver_inserted": 0, "silver_updated"

'{"batch_id": "run_20261117T184549Z_4692b9_backfill", "records": [{"repository": "surrealdb/surrealdb", "entity": "commits", "bronze_rows": 393, "silver_inserted": 0, "silver_updated": 393, "quarantined_rows": 0, "excluded_prs": 0, "reconciled": true}, {"repository": "surrealdb/surrealdb", "entity": "issues", "bronze_rows": 177, "silver_inserted": 0, "silver_updated": 100, "quarantined_rows": 0, "excluded_prs": 77, "reconciled": true}, {"repository": "surrealdb/surrealdb", "entity": "pulls", "bronze_rows": 44, "silver_inserted": 0, "silver_updated": 44, "quarantined_rows": 0, "excluded_prs": 0, "reconciled": true}, {"repository": "surrealdb/surrealdb", "entity": "releases", "bronze_rows": 9, "silver_inserted": 0, "silver_updated": 9, "quarantined_rows": 0, "excluded_prs": 0, "reconciled": true}, {"repository": "surrealdb/surrealdb", "entity": "repo_metadata", "bronze_rows": 0, "silver_inserted": 0, "silver_updated": 0, "quarantined_rows": 0, "excluded_prs": 0, "reconciled": true}, {"re

In [0]:
# ── B-4 : Gold pass — refresh star schema after backfill ────────────────────
print(f"\n[B-4] Gold — refresh star schema")
run_nb(NB_05, {
    "catalog":        CATALOG,
    "batch_id":       B_BATCH_ID,
    "gold_mode":      "full",
    "analysis_since": "2026-04-01",
    "analysis_until": "2026-09-30",
    "run_optimize":   "true",
})


[B-4] Gold — refresh star schema
  ✓ 05_silver_to_gold  → '{"batch_id": "run_20261117T184549Z_4692b9_backfill", "status": "SUCCESS", "gold_mode": "full", "tables_processed": ["dim_repo", "dim_date", "dim_actor", "fact_commits", "fact_issues", "fact_pull_requests", "fact_releases", "fact_repo_snapshot", "fact_repo_daily_activity", "fact_repo_stars_weekly"], "quality_gates": "ALL_PASSED", "reconciliation": [{"table": "fact_commits", "silver_rows": 19042, "gold_rows": 19042, "delta": 0, "status": "PASS"}, {"table": "fact_issues", "silver_rows": 21751, "gold_rows": 21751, "delta": 0, "status": "PASS"}, {"table": "fact_pull_requests", "silver_rows": 13426, "gold_rows": 13426, "delta": 0, "status": "PASS"}, {"table": "fact_releases", "silver_rows": 795, "gold_rows": 795, "delta": 0, "status": "PASS"}, {"table": "fact_repo_snapshot", "silver_rows": 10, "gold_rows": 10, "delta": 0, "status": "PASS"}]}'  (232s)


'{"batch_id": "run_20261117T184549Z_4692b9_backfill", "status": "SUCCESS", "gold_mode": "full", "tables_processed": ["dim_repo", "dim_date", "dim_actor", "fact_commits", "fact_issues", "fact_pull_requests", "fact_releases", "fact_repo_snapshot", "fact_repo_daily_activity", "fact_repo_stars_weekly"], "quality_gates": "ALL_PASSED", "reconciliation": [{"table": "fact_commits", "silver_rows": 19042, "gold_rows": 19042, "delta": 0, "status": "PASS"}, {"table": "fact_issues", "silver_rows": 21751, "gold_rows": 21751, "delta": 0, "status": "PASS"}, {"table": "fact_pull_requests", "silver_rows": 13426, "gold_rows": 13426, "delta": 0, "status": "PASS"}, {"table": "fact_releases", "silver_rows": 795, "gold_rows": 795, "delta": 0, "status": "PASS"}, {"table": "fact_repo_snapshot", "silver_rows": 10, "gold_rows": 10, "delta": 0, "status": "PASS"}]}'

In [0]:
# ── B-5 : Verification ───────────────────────────────────────────────────────
show_counts(f"after SECTION B — BACKFILL {B_REPO} [{B_SINCE} → {B_UNTIL}]")
show_logs("after SECTION B", n=15)


ROW COUNTS  after SECTION B — BACKFILL surrealdb/surrealdb [2026-04-01T00:00:00Z → 2026-06-30T23:59:59Z]
  workspace.bronze.commits                          19,042 rows
  workspace.bronze.issues                           40,229 rows
  workspace.bronze.pull_requests                    13,426 rows
  workspace.bronze.releases                            795 rows
  workspace.bronze.repo_metadata                        10 rows
  workspace.bronze.stars_history                     5,981 rows
  workspace.silver.commits                          19,042 rows
  workspace.silver.issues                           21,751 rows
  workspace.silver.pull_requests                    13,426 rows
  workspace.silver.releases                            795 rows
  workspace.silver.repo_metadata                        10 rows
  workspace.silver.stars_weekly                      5,981 rows
  workspace.gold.dim_repo                             10 rows
  workspace.gold.dim_date                          7,305 rows
  

---
## ══════════════════════════════════════════════════════
## SECTION C — DRIFT TEST
## ══════════════════════════════════════════════════════
**Purpose:** Validate that the pipeline correctly handles schema drift:
- **Anomaly A:** New unexpected top-level field `new_field_test` on every record.
  → Bronze absorbs it via schema evolution (evolve_table_schema), adding a new STRING column.
- **Anomaly B:** `comments` field set to string `"many"` on 3 issue records.
  → Bronze isolates and quarantines those 3 rows in `ops.silver_quarantine` with layer='bronze'.
- **Anomaly C:** `created_at = "not-a-date"` on 1 true issue record (index 4).
  → Silver quarantines that 1 row in `ops.silver_quarantine` with layer='silver'.

**Pre-requisite:** Run `python scripts/make_drift_sample.py` locally and upload
`samples/drift/` to the Volume at `{base_path}/drift/driftlab_surrealdb/`.

In [0]:
# ── C-1 : Configure drift test ───────────────────────────────────────────────
C_REPO        = "driftlab/surrealdb"
C_SINCE       = "2026-01-01T00:00:00Z"   # wide window — catches all drift records
C_UNTIL       = ""
C_RUN_MODE    = "full"
C_BATCH_ID    = f"{BATCH_ID}_drift"
C_DRIFT_PATH  = f"{BASE_PATH}/drift"     # Volume path where drift samples are staged

print(f"[Section C] DRIFT TEST")
print(f"  repo              : {C_REPO}")
print(f"  drift_path        : {C_DRIFT_PATH}")
print(f"  batch_id          : {C_BATCH_ID}")
print()
print("  Pre-requisite check — drift files must exist on the Volume:")
for entity in ("issues", "commits"):
    path = f"{C_DRIFT_PATH}/driftlab_surrealdb/{entity}.json"
    try:
        dbutils.fs.ls(path)
        print(f"    ✓ {path}")
    except Exception:
        print(f"    ✗ MISSING: {path}  ← run make_drift_sample.py then upload")

[Section C] DRIFT TEST
  repo              : driftlab/surrealdb
  drift_path        : /Volumes/workspace/bronze_data/raw/drift
  batch_id          : run_20261117T184549Z_4692b9_drift

  Pre-requisite check — drift files must exist on the Volume:
    ✓ /Volumes/workspace/bronze_data/raw/drift/driftlab_surrealdb/issues.json
    ✓ /Volumes/workspace/bronze_data/raw/drift/driftlab_surrealdb/commits.json


In [0]:
# ── C-2 : Bronze pass using drift data ──────────────────────────────────────
# Point base_path to the drift folder so notebook 02 reads the injected files
print(f"\n[C-2] Bronze (drift) — issues & commits for {C_REPO}")
for entity in ("issues", "commits"):
    run_nb(NB_02, {
        "catalog":   CATALOG,
        "repo":      C_REPO,
        "entity":    entity,
        "base_path": C_DRIFT_PATH,          # ← drift Volume path
        "since":     C_SINCE,
        "until":     C_UNTIL,
        "batch_id":  C_BATCH_ID,
        "run_mode":  C_RUN_MODE,
    })


[C-2] Bronze (drift) — issues & commits for driftlab/surrealdb
  ✓ 02_raw_to_bronze  → '{"batch_id": "run_20261117T184549Z_4692b9_drift", "total_inserted": 2, "total_updated": 0, "results": [{"repo": "driftlab/surrealdb", "entity": "issues", "status": "SUCCESS", "inserted": 2, "updated": 0}]}'  (41s)
  ✓ 02_raw_to_bronze  → '{"batch_id": "run_20261117T184549Z_4692b9_drift", "total_inserted": 5, "total_updated": 0, "results": [{"repo": "driftlab/surrealdb", "entity": "commits", "status": "SUCCESS", "inserted": 5, "updated": 0}]}'  (31s)


In [0]:
# ── C-3 : Silver pass — expect quarantine rows ───────────────────────────────
print(f"\n[C-3] Silver (drift) — {C_REPO}")
run_nb(NB_03, {
    "catalog":  CATALOG,
    "repo":     C_REPO,
    "since":    C_SINCE,
    "until":    C_UNTIL,
    "batch_id": C_BATCH_ID,
    "run_mode": C_RUN_MODE,
})


[C-3] Silver (drift) — driftlab/surrealdb
  ✓ 03_bronze_to_silver  → '{"batch_id": "run_20261117T184549Z_4692b9_drift", "records": [{"repository": "driftlab/surrealdb", "entity": "commits", "bronze_rows": 5, "silver_inserted": 5, "silver_updated": 0, "quarantined_rows": 0, "excluded_prs": 0, "reconciled": true}, {"repository": "driftlab/surrealdb", "entity": "issues", "bronze_rows": 2, "silver_inserted": 0, "silver_updated": 0, "quarantined_rows": 1, "excluded_prs": 1, "reconciled": true}, {"repository": "driftlab/surrealdb", "entity": "pulls", "bronze_rows": 0, "silver_inserted": 0, "silver_updated": 0, "quarantined_rows": 0, "excluded_prs": 0, "reconciled": true}, {"repository": "driftlab/surrealdb", "entity": "releases", "bronze_rows": 0, "silver_inserted": 0, "silver_updated": 0, "quarantined_rows": 0, "excluded_prs": 0, "reconciled": true}, {"repository": "driftlab/surrealdb", "entity": "repo_metadata", "bronze_rows": 0, "silver_inserted": 0, "silver_updated": 0, "quarantined_row

'{"batch_id": "run_20261117T184549Z_4692b9_drift", "records": [{"repository": "driftlab/surrealdb", "entity": "commits", "bronze_rows": 5, "silver_inserted": 5, "silver_updated": 0, "quarantined_rows": 0, "excluded_prs": 0, "reconciled": true}, {"repository": "driftlab/surrealdb", "entity": "issues", "bronze_rows": 2, "silver_inserted": 0, "silver_updated": 0, "quarantined_rows": 1, "excluded_prs": 1, "reconciled": true}, {"repository": "driftlab/surrealdb", "entity": "pulls", "bronze_rows": 0, "silver_inserted": 0, "silver_updated": 0, "quarantined_rows": 0, "excluded_prs": 0, "reconciled": true}, {"repository": "driftlab/surrealdb", "entity": "releases", "bronze_rows": 0, "silver_inserted": 0, "silver_updated": 0, "quarantined_rows": 0, "excluded_prs": 0, "reconciled": true}, {"repository": "driftlab/surrealdb", "entity": "repo_metadata", "bronze_rows": 0, "silver_inserted": 0, "silver_updated": 0, "quarantined_rows": 0, "excluded_prs": 0, "reconciled": true}, {"repository": "driftla

In [0]:
# ── C-4 : Verification ───────────────────────────────────────────────────────
show_counts("after SECTION C — DRIFT TEST")
show_logs("after SECTION C", n=10)

# ── C-4b : Drift-specific assertions ────────────────────────────────────────
print("\n[C-4b] Drift-specific assertions")

# 1. Bronze table must have evolved to include new_field_test
bronze_issues_cols = [c.name for c in spark.table(f"{CATALOG}.bronze.issues").schema]
bronze_commits_cols = [c.name for c in spark.table(f"{CATALOG}.bronze.commits").schema]
assert "new_field_test" in bronze_issues_cols, \
    "FAIL: 'new_field_test' not found in bronze.issues — schema evolution did not add the column"
assert "new_field_test" in bronze_commits_cols, \
    "FAIL: 'new_field_test' not found in bronze.commits — schema evolution did not add the column"
print(f"  ✓ bronze.issues and bronze.commits have 'new_field_test' column  (schema evolution confirmed)")

# 2. Quarantine rows for this batch with layer='bronze' and entity='issues' == 3
bronze_q = (
    spark.table(f"{CATALOG}.ops.silver_quarantine")
         .filter(f"batch_id = '{C_BATCH_ID}' AND layer = 'bronze' AND entity = 'issues'")
)
bronze_q_count = bronze_q.count()
assert bronze_q_count == 3, \
    f"FAIL: Expected == 3 bronze issues quarantine rows for batch {C_BATCH_ID}, got {bronze_q_count}"
print(f"  ✓ ops.silver_quarantine has {bronze_q_count} row(s) with layer='bronze' and entity='issues' (== 3 expected)")

# 3. Quarantine rows for this batch with layer='silver' and entity='issues' >= 1
silver_q = (
    spark.table(f"{CATALOG}.ops.silver_quarantine")
         .filter(f"batch_id = '{C_BATCH_ID}' AND layer = 'silver' AND entity = 'issues'")
)
silver_q_count = silver_q.count()
assert silver_q_count >= 1, \
    f"FAIL: Expected >= 1 silver issues quarantine rows for batch {C_BATCH_ID}, got {silver_q_count}"
print(f"  ✓ ops.silver_quarantine has {silver_q_count} row(s) with layer='silver' and entity='issues' (>= 1 expected)")

# 4. Log entry must record drift detection in parameter
drift_logs = (
    spark.table(f"{CATALOG}.ops.pipeline_execution_logs")
         .filter(f"batch_id = '{C_BATCH_ID}' AND layer = 'BRONZE'")
         .filter("parameter LIKE '%drift_keys=%' AND parameter LIKE '%new_field_test%'")
)
assert drift_logs.count() > 0, \
    f"FAIL: ops.pipeline_execution_logs row for batch {C_BATCH_ID} does not contain drift_keys with 'new_field_test'"
print("  ✓ ops.pipeline_execution_logs has Bronze log entry with drift_keys containing 'new_field_test'")

print("\n  All drift assertions passed.")
silver_q.select(
    "batch_id", "entity", "rejection_reason", "raw_payload"
).show(10, truncate=100)


ROW COUNTS  after SECTION C — DRIFT TEST
  workspace.bronze.commits                          19,047 rows
  workspace.bronze.issues                           40,231 rows
  workspace.bronze.pull_requests                    13,426 rows
  workspace.bronze.releases                            795 rows
  workspace.bronze.repo_metadata                        10 rows
  workspace.bronze.stars_history                     5,981 rows
  workspace.silver.commits                          19,047 rows
  workspace.silver.issues                           21,751 rows
  workspace.silver.pull_requests                    13,426 rows
  workspace.silver.releases                            795 rows
  workspace.silver.repo_metadata                        10 rows
  workspace.silver.stars_weekly                      5,981 rows
  workspace.gold.dim_repo                             10 rows
  workspace.gold.dim_date                          7,305 rows
  workspace.gold.dim_actor                         3,603 rows
  wo

In [0]:
# ── C-5 : Cleanup drift data from bronze/silver tables ───────────────────────
print(f"\n[C-5] Cleanup {C_REPO} rows from Bronze and Silver")
for tbl in ["issues", "commits"]:
    try:
        spark.sql(f"DELETE FROM {CATALOG}.bronze.{tbl} WHERE repo_full_name = '{C_REPO}'")
    except Exception as e:
        print(f"  Could not delete from bronze.{tbl}: {e}")

for tbl in ["issues", "commits", "pull_requests"]:
    try:
        spark.sql(f"DELETE FROM {CATALOG}.silver.{tbl} WHERE repo_full_name = '{C_REPO}'")
    except Exception as e:
        print(f"  Could not delete from silver.{tbl}: {e}")

print(f"  ✓ Cleaned up any rows for {C_REPO} from bronze and silver (quarantine and execution logs retained)")


[C-5] Cleanup driftlab/surrealdb rows from Bronze and Silver
  ✓ Cleaned up any rows for driftlab/surrealdb from bronze and silver (quarantine and execution logs retained)


In [0]:
# ── C-6 : Regression Guard — Normal Bronze Load After Drift ──────────────────
print(f"\n[C-6] Regression guard: Normal Bronze load after drift schema evolution")
# Runs a standard Bronze load on an evolved table (issues for redis/redis) without drift
run_nb(NB_02, {
    "catalog":   CATALOG,
    "repo":      "redis/redis",
    "entity":    "issues",
    "base_path": BASE_PATH,
    "since":     "",
    "until":     "",
    "batch_id":  f"{C_BATCH_ID}_regression_guard",
    "run_mode":  "full",
})
print("  ✓ Regression guard passed: normal load succeeded against evolved table")


[C-6] Regression guard: Normal Bronze load after drift schema evolution
  ✓ 02_raw_to_bronze  → '{"batch_id": "run_20261117T184549Z_4692b9_drift_regression_guard", "total_inserted": 2, "total_updated": 1516, "results": [{"repo": "redis/redis", "entity": "issues", "status": "SUCCESS", "inserted": 2, "updated": 1516}]}'  (41s)
  ✓ Regression guard passed: normal load succeeded against evolved table


---
## ══════════════════════════════════════════════════════
## SECTION D — IDEMPOTENCY PROOF
## ══════════════════════════════════════════════════════
**Purpose:** Run the **same batch** twice (same `batch_id`) and assert that
row counts in Bronze, Silver, and the quarantine table are **identical** after
both passes.  This proves the `MERGE INTO` logic is truly idempotent.

In [0]:
# ── D-1 : Configure idempotency test ────────────────────────────────────────
# Reuses the same incremental window so the proof runs on real fetched data.
# D_REPO is intentionally hardcoded to a single repo to keep the proof fast.
D_REPO      = "redis/redis"        # fixed: single repo for the proof
D_SINCE     = INCR_SINCE           # widget: incr_since (same window as Section A)
D_UNTIL     = INCR_UNTIL           # widget: incr_until
D_RUN_MODE  = "incremental"
D_BATCH_ID  = f"{BATCH_ID}_idempotency_proof"   # fixed ID — MUST be the same on both runs

def _pipeline_state():
    """Collect current row counts and metric sums from Bronze, Silver, Ops, and Gold."""
    state = {}
    layers = [
        (f"{CATALOG}.bronze", ["commits", "issues", "pull_requests", "releases", "repo_metadata", "stars_history"]),
        (f"{CATALOG}.silver", ["commits", "issues", "pull_requests", "releases", "repo_metadata", "stars_weekly"]),
        (f"{CATALOG}.gold",   [
            "dim_repo", "dim_date", "dim_actor",
            "fact_commits", "fact_issues", "fact_pull_requests",
            "fact_releases", "fact_repo_snapshot", "fact_repo_daily_activity", "fact_repo_stars_weekly"
        ]),
        (f"{CATALOG}.ops",    ["silver_quarantine"]),
    ]
    for schema, tables in layers:
        for tbl in tables:
            key = f"{schema}.{tbl}"
            try:
                state[key] = spark.table(key).count()
            except Exception:
                state[key] = None

    # Collect analytical measure sums to prove aggregate stability
    try:
        agg_sums = (
            spark.table(f"{CATALOG}.gold.fact_repo_daily_activity")
            .select(
                F.sum("commits_total").alias("sum_commits_total"),
                F.sum("commits_human").alias("sum_commits_human"),
                F.sum("commits_bot").alias("sum_commits_bot"),
            )
            .first()
        )
        if agg_sums:
            state["gold.fact_repo_daily_activity[sum_commits_total]"] = agg_sums["sum_commits_total"]
            state["gold.fact_repo_daily_activity[sum_commits_human]"] = agg_sums["sum_commits_human"]
            state["gold.fact_repo_daily_activity[sum_commits_bot]"] = agg_sums["sum_commits_bot"]
    except Exception:
        pass

    return state

print(f"[Section D] IDEMPOTENCY PROOF")
print(f"  repo      : {D_REPO}")
print(f"  since     : {D_SINCE}")
print(f"  batch_id  : {D_BATCH_ID}  (same for both runs)")

[Section D] IDEMPOTENCY PROOF
  repo      : redis/redis
  since     : 2026-09-25T00:00:00Z
  batch_id  : run_20261117T184549Z_4692b9_idempotency_proof  (same for both runs)


In [0]:
# ── D-2 : FIRST pass ─────────────────────────────────────────────────────────
print("\n[D-2] First pass — Bronze")
for entity in ALL_ENTITIES:
    run_nb(NB_02, {
        "catalog":   CATALOG,
        "repo":      D_REPO,
        "entity":    entity,
        "base_path": BASE_PATH,
        "since":     D_SINCE,
        "until":     D_UNTIL,
        "batch_id":  D_BATCH_ID,
        "run_mode":  D_RUN_MODE,
    })

print("\n[D-2] First pass — Silver")
run_nb(NB_03, {
    "catalog":  CATALOG,
    "repo":     D_REPO,
    "since":    D_SINCE,
    "until":    D_UNTIL,
    "batch_id": D_BATCH_ID,
    "run_mode": D_RUN_MODE,
})

print("\n[D-2] First pass — Gold")
run_nb(NB_05, {
    "catalog":        CATALOG,
    "batch_id":       D_BATCH_ID,
    "gold_mode":      "incremental",
    "analysis_since": "2026-04-01",
    "analysis_until": "2026-09-30",
    "run_optimize":   "false",
})

state_after_first = _pipeline_state()
print("\nPipeline state after first pass:")
for k, v in state_after_first.items():
    print(f"  {k:<55} {v}")


[D-2] First pass — Bronze
  ✓ 02_raw_to_bronze  → '{"batch_id": "run_20261117T184549Z_4692b9_idempotency_proof", "total_inserted": 0, "total_updated": 1, "results": [{"repo": "redis/redis", "entity": "repo_metadata", "status": "SUCCESS", "inserted": 0, "updated": 1}]}'  (31s)
  ✓ 02_raw_to_bronze  → '{"batch_id": "run_20261117T184549Z_4692b9_idempotency_proof", "total_inserted": 0, "total_updated": 147, "results": [{"repo": "redis/redis", "entity": "commits", "status": "SUCCESS", "inserted": 0, "updated": 147}]}'  (31s)
  ✓ 02_raw_to_bronze  → '{"batch_id": "run_20261117T184549Z_4692b9_idempotency_proof", "total_inserted": 0, "total_updated": 569, "results": [{"repo": "redis/redis", "entity": "issues", "status": "SUCCESS", "inserted": 0, "updated": 569}]}'  (31s)
  ✓ 02_raw_to_bronze  → '{"batch_id": "run_20261117T184549Z_4692b9_idempotency_proof", "total_inserted": 0, "total_updated": 410, "results": [{"repo": "redis/redis", "entity": "pulls", "status": "SUCCESS", "inserted": 0, "upd

In [0]:
# ── D-3 : SECOND pass (identical params) ────────────────────────────────────
print("\n[D-3] Second pass — Bronze  (same batch_id, same data)")
for entity in ALL_ENTITIES:
    run_nb(NB_02, {
        "catalog":   CATALOG,
        "repo":      D_REPO,
        "entity":    entity,
        "base_path": BASE_PATH,
        "since":     D_SINCE,
        "until":     D_UNTIL,
        "batch_id":  D_BATCH_ID,
        "run_mode":  D_RUN_MODE,
    })

print("\n[D-3] Second pass — Silver  (same batch_id, same data)")
run_nb(NB_03, {
    "catalog":  CATALOG,
    "repo":     D_REPO,
    "since":    D_SINCE,
    "until":    D_UNTIL,
    "batch_id": D_BATCH_ID,
    "run_mode": D_RUN_MODE,
})

print("\n[D-3] Second pass — Gold  (same batch_id, same data)")
run_nb(NB_05, {
    "catalog":        CATALOG,
    "batch_id":       D_BATCH_ID,
    "gold_mode":      "incremental",
    "analysis_since": "2026-04-01",
    "analysis_until": "2026-09-30",
    "run_optimize":   "false",
})

state_after_second = _pipeline_state()
print("\nPipeline state after second pass:")
for k, v in state_after_second.items():
    print(f"  {k:<55} {v}")


[D-3] Second pass — Bronze  (same batch_id, same data)
  ✓ 02_raw_to_bronze  → '{"batch_id": "run_20261117T184549Z_4692b9_idempotency_proof", "total_inserted": 0, "total_updated": 1, "results": [{"repo": "redis/redis", "entity": "repo_metadata", "status": "SUCCESS", "inserted": 0, "updated": 1}]}'  (31s)
  ✓ 02_raw_to_bronze  → '{"batch_id": "run_20261117T184549Z_4692b9_idempotency_proof", "total_inserted": 0, "total_updated": 147, "results": [{"repo": "redis/redis", "entity": "commits", "status": "SUCCESS", "inserted": 0, "updated": 147}]}'  (21s)
  ✓ 02_raw_to_bronze  → '{"batch_id": "run_20261117T184549Z_4692b9_idempotency_proof", "total_inserted": 0, "total_updated": 569, "results": [{"repo": "redis/redis", "entity": "issues", "status": "SUCCESS", "inserted": 0, "updated": 569}]}'  (33s)
  ✓ 02_raw_to_bronze  → '{"batch_id": "run_20261117T184549Z_4692b9_idempotency_proof", "total_inserted": 0, "total_updated": 410, "results": [{"repo": "redis/redis", "entity": "pulls", "status": "

In [0]:
# ── D-4 : Assert idempotency ─────────────────────────────────────────────────
print("\n[D-4] Idempotency assertion")
all_pass = True
for key in state_after_first:
    v1 = state_after_first[key]
    v2 = state_after_second[key]
    if v1 is None and v2 is None:
        continue
    if v1 != v2:
        print(f"  ✗ FAIL: {key}  first={v1}  second={v2}  DELTA={(v2 or 0) - (v1 or 0)}")
        all_pass = False
    else:
        print(f"  ✓ PASS: {key}  value={v2} (unchanged)")

if all_pass:
    print("\n  ✓ FULL MEDALLION IDEMPOTENCY CONFIRMED:")
    print("    All table row counts and measure sums in Bronze, Silver, and Gold are identical after the second pass.")
    print("    Delta MERGE INTO logic correctly deduplicates and upserts across the entire pipeline.\n")
else:
    raise AssertionError(
        "Idempotency check FAILED. One or more table row counts or measure sums changed on the second pass. "
        "Review the MERGE keys in notebooks 02, 03, and 05."
    )

show_counts("after SECTION D — IDEMPOTENCY PROOF")
show_logs("after SECTION D", n=15)


[D-4] Idempotency assertion
  ✓ PASS: workspace.bronze.commits  value=19042 (unchanged)
  ✓ PASS: workspace.bronze.issues  value=40231 (unchanged)
  ✓ PASS: workspace.bronze.pull_requests  value=13426 (unchanged)
  ✓ PASS: workspace.bronze.releases  value=795 (unchanged)
  ✓ PASS: workspace.bronze.repo_metadata  value=10 (unchanged)
  ✓ PASS: workspace.bronze.stars_history  value=5981 (unchanged)
  ✓ PASS: workspace.silver.commits  value=19042 (unchanged)
  ✓ PASS: workspace.silver.issues  value=21751 (unchanged)
  ✓ PASS: workspace.silver.pull_requests  value=13426 (unchanged)
  ✓ PASS: workspace.silver.releases  value=795 (unchanged)
  ✓ PASS: workspace.silver.repo_metadata  value=10 (unchanged)
  ✓ PASS: workspace.silver.stars_weekly  value=5981 (unchanged)
  ✓ PASS: workspace.gold.dim_repo  value=10 (unchanged)
  ✓ PASS: workspace.gold.dim_date  value=7305 (unchanged)
  ✓ PASS: workspace.gold.dim_actor  value=3603 (unchanged)
  ✓ PASS: workspace.gold.fact_commits  value=19042 (unc